# Lab 2: Pre-trained CNNs, transfer learning and fine-tuning
**Owner: team leader.** Skeleton notebook: no results yet, no training inside the notebook.

**Goal:** reuse torchvision models pre-trained on ImageNet (ResNet18, DenseNet121; VGG optional) for a 10-class image task: replace the final layer, compare frozen backbone vs fine-tuning, vary learning rate and batch size, monitor with TensorBoard.

**Dataset:** CIFAR-10 (ASSUMPTION, waiting for the lecturer to confirm). Task: Practice 2, Chapter 3 page 23.

---

**Tiếng Việt**

# Lab 2: CNN pre-trained, transfer learning và fine-tuning
**Phụ trách: nhóm trưởng.** Notebook khung: chưa có kết quả, không huấn luyện trong notebook.

**Mục tiêu:** dùng lại model torchvision pre-trained trên ImageNet (ResNet18, DenseNet121; VGG tùy chọn) cho bài toán ảnh 10 lớp: thay lớp cuối, so sánh đóng băng và fine-tune, đổi learning rate và batch size, theo dõi bằng TensorBoard.

**Dữ liệu:** CIFAR-10 (GIẢ ĐỊNH, chờ cô xác nhận). Đề: Practice 2, Chương 3 trang 23.

## 2. Pipeline
**Owner: team leader.**

| Step | Module (owner) | Functions | Product |
|---|---|---|---|
| EDA | `eda` (A) | `run_eda` | class counts, sample grid, channel mean/std |
| Preprocessing | `preprocessing` (A) | `load_raw_dataset`, `make_split`, `save_split`, `load_split` | train/val split JSON |
| Features | `features` (A) | `build_transforms`, `make_loaders` | loaders returning `[B,3,H,W]` |
| Model | `model` (A) | `build_model`, `describe_model`, `unfreeze_last_blocks` | logits `[B,10]`, parameter counts |
| Train | `train` (B) | `train_model`, `save_checkpoint`, `load_checkpoint` | checkpoints, TensorBoard logs |
| Evaluate | `evaluate` (B) | `evaluate`, `confusion_matrix`, `top_confusions` | accuracy, confusion matrix |
| Experiment | `experiment` (B) | `run_comparison`, `save_results` | comparison CSV |

---

**Tiếng Việt**

## 2. Pipeline
**Phụ trách: nhóm trưởng.** Bảng giống phần tiếng Anh: mỗi bước gồm module (người làm), hàm gọi và sản phẩm.

## 3. Setup
**Owner: team leader.** Find the repo root (as in the Lab 1 notebook), import modules, choose device, set the seed.

---

**Tiếng Việt**

## 3. Thiết lập
**Phụ trách: nhóm trưởng.** Tìm thư mục gốc repo (như notebook Lab 1), import module, chọn thiết bị, cố định seed.

In [ ]:
# Owner: team leader. Uncomment when the modules exist and pass their smoke tests.
# import sys
# from pathlib import Path
# import torch
#
# # Find the repo root (folder that contains both shared/ and labs/), same as the Lab 1 notebook
# cwd = Path.cwd().resolve()
# repo_root = next((p for p in [cwd, *cwd.parents] if (p / "shared").is_dir() and (p / "labs").is_dir()), cwd)
# if str(repo_root) not in sys.path:
#     sys.path.insert(0, str(repo_root))
#
# from shared.paths import raw_data_dir, artifact_dir
# from shared.seed import set_seed
# from labs.lab2.eda.eda import run_eda
# from labs.lab2.preprocessing.preprocessing import load_raw_dataset, make_split, save_split, load_split
# from labs.lab2.features.features import build_transforms, make_loaders
# from labs.lab2.model.model import build_model, describe_model, unfreeze_last_blocks
# from labs.lab2.train.train import train_model, save_checkpoint, load_checkpoint
# from labs.lab2.evaluate.evaluate import evaluate, confusion_matrix, top_confusions
# from labs.lab2.experiment.experiment import run_comparison, save_results
#
# SEED = 42
# set_seed(SEED)
# device = "cuda" if torch.cuda.is_available() else "cpu"
# print("repo root:", repo_root, "| device:", device)


## 4. Data: EDA, preprocessing, features
**Owner: Member A.** Calls: `run_eda`, `load_raw_dataset`, `make_split` / `load_split`, `build_transforms`, `make_loaders`.

Show: number of images, class counts, a sample grid, the split sizes, one batch shape `[B,3,H,W]`. Write the chosen image size and whether a subset is used (CPU constraint). Data loading uses `num_workers=0`.

---

**Tiếng Việt**

## 4. Dữ liệu: EDA, tiền xử lý, feature
**Phụ trách: Member A.** Gọi: `run_eda`, `load_raw_dataset`, `make_split` / `load_split`, `build_transforms`, `make_loaders`.

Hiển thị: số ảnh, số lượng mỗi lớp, lưới ảnh mẫu, kích thước các tập, shape một batch `[B,3,H,W]`. Ghi rõ kích thước ảnh và có dùng tập con không (vì chỉ có CPU). Nạp dữ liệu với `num_workers=0`.

In [ ]:
# Owner: Member A
# eda_stats = run_eda(str(raw_data_dir('lab2')), str(artifact_dir('lab2', 'figures')))
# train_set, test_set = load_raw_dataset(str(raw_data_dir('lab2')), download=True)
# train_idx, val_idx = load_split(str(artifact_dir('lab2', 'splits') / 'train_val_split.json'))
# train_tf, eval_tf = build_transforms(image_size=..., augment=False)
# train_loader, val_loader, test_loader = make_loaders(train_set, test_set, train_idx, val_idx, train_tf, eval_tf, batch_size=...)

## 5. Model
**Owner: Member A.** Calls: `build_model`, `describe_model`, `unfreeze_last_blocks`.

Print each model's structure, total vs trainable parameters (frozen and fine-tune settings), and the replaced final layer.

---

**Tiếng Việt**

## 5. Mô hình
**Phụ trách: Member A.** Gọi: `build_model`, `describe_model`, `unfreeze_last_blocks`.

In cấu trúc từng model, số tham số tổng và số tham số được học (khi đóng băng và khi fine-tune), và lớp cuối đã thay.

In [ ]:
# Owner: Member A
# for name in ['resnet18', 'densenet121']:
#     m = build_model(name, num_classes=10, freeze_backbone=True, pretrained=True)
#     print(name, describe_model(m))

## 6. Training (run by script, the notebook only loads results)
**Owner: Member B.** Training is done with `python -m labs.lab2.main` (calls `train_model` / `run_comparison`). Here we only load the saved history, checkpoints (`load_checkpoint`) and show TensorBoard screenshots or `tensorboard --logdir <log folder>`.

---

**Tiếng Việt**

## 6. Huấn luyện (chạy bằng script, notebook chỉ nạp kết quả)
**Phụ trách: Member B.** Huấn luyện bằng `python -m labs.lab2.main` (gọi `train_model` / `run_comparison`). Ở đây chỉ nạp history, checkpoint đã lưu (`load_checkpoint`) và hiển thị ảnh chụp TensorBoard hoặc `tensorboard --logdir <thư mục log>`.

In [ ]:
# Owner: Member B
from pathlib import Path
import torch
from shared.paths import artifact_dir
from labs.lab2.train.train import load_checkpoint
from labs.lab2.model.model import build_model

models_dir = artifact_dir("lab2", "models")
ckpt_files = sorted(list(Path(models_dir).glob("*.pt")))

if ckpt_files:
    best_ckpt = ckpt_files[0]
    print(f"[INFO] Loading checkpoint: {best_ckpt}")
    model = build_model("resnet18", num_classes=10, freeze_backbone=True, pretrained=False)
    model = load_checkpoint(model, str(best_ckpt), device=device)
    print("[SUCCESS] Checkpoint loaded successfully into eval mode on", device)
else:
    print(f"[NOTE] No checkpoint found yet in {models_dir}.")
    print("Run training via python -m labs.lab2.main or train_model() to produce checkpoints.")


## 7. Evaluation on validation
**Owner: Member B.** Calls: `evaluate`, `confusion_matrix` on `val_loader`. All choices (model, freeze/fine-tune, lr, batch size, epochs) are made here, never on test.

---

**Tiếng Việt**

## 7. Đánh giá trên validation
**Phụ trách: Member B.** Gọi: `evaluate`, `confusion_matrix` trên `val_loader`. Mọi lựa chọn (model, đóng băng/fine-tune, lr, batch size, số epoch) làm ở đây, không bao giờ trên test.

In [ ]:
# Owner: Member B
from labs.lab2.evaluate.evaluate import evaluate, confusion_matrix

if "model" in locals() and "val_loader" in locals():
    val_loss, val_acc = evaluate(model, val_loader, device)
    cm = confusion_matrix(model, val_loader, device, num_classes=10)
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_acc * 100:.2f}%")
    print(f"Confusion Matrix shape: {cm.shape}")
else:
    print("[NOTE] model or val_loader not initialized. Run preceding sections first.")


## 8. Predictions and errors
**Owner: Member B.** Call: `top_confusions(cm, class_names, k=5)`. Show some predicted vs actual images. Reasons for errors are hypotheses unless tested.

---

**Tiếng Việt**

## 8. Dự đoán và lỗi
**Phụ trách: Member B.** Gọi: `top_confusions(cm, class_names, k=5)`. Hiển thị vài ảnh dự đoán so với thực tế. Lý do gây lỗi chỉ là giả thuyết nếu chưa kiểm chứng.

In [ ]:
# Owner: Member B
from labs.lab2.evaluate.evaluate import top_confusions

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

if "cm" in locals():
    print("Top 5 most frequent confusion pairs (True -> Predicted: Count):")
    top_5 = top_confusions(cm, class_names, k=5)
    for true_c, pred_c, count in top_5:
        print(f"  - {true_c:12s} -> {pred_c:12s}: {count} errors")
else:
    print("[NOTE] Confusion matrix `cm` not found. Run section 7 first.")


## 9. Experiments
**Owner: Member B.** Calls: `run_comparison`, `save_results` (results loaded from the CSV). Compare: 2 models; frozen vs fine-tune; 2+ learning rates; 2+ batch sizes.

**Noise:** with n validation images, the standard error of accuracy is about sqrt(p(1-p)/n) (e.g. about 0.4 points for p = 0.89, n = 5,000, i.e. 10% of 50,000). With one seed per setting, differences smaller than about 2 standard errors are not evidence that one setting is better.

---

**Tiếng Việt**

## 9. Thí nghiệm
**Phụ trách: Member B.** Gọi: `run_comparison`, `save_results` (nạp kết quả từ CSV). So sánh: 2 model; đóng băng và fine-tune; từ 2 learning rate; từ 2 batch size.

**Sai số:** với n ảnh validation, sai số chuẩn của accuracy khoảng sqrt(p(1-p)/n) (vd khoảng 0,4 điểm % khi p = 0,89, n = 5.000, tức 10% của 50.000). Mỗi cấu hình chỉ chạy một seed thì chênh lệch nhỏ hơn khoảng 2 lần sai số chuẩn chưa chứng minh được cấu hình nào tốt hơn.

In [ ]:
# Owner: Member B
import pandas as pd
from pathlib import Path
from shared.paths import artifact_dir

runs_dir = Path(artifact_dir("lab2", "runs"))
csv_files = list(runs_dir.glob("*.csv"))
if csv_files:
    results_csv = csv_files[0]
    print(f"[INFO] Loading comparison table from {results_csv}:")
    df = pd.read_csv(results_csv)
    display(df)
else:
    print(f"[NOTE] No results CSV found in {runs_dir}. Run experiment pipeline first.")


## 10. Final test (not used to choose the model)
**Owner: Member B.** Call: `evaluate(model, test_loader, device)` for the setting chosen in sections 7 and 9 only.

---

**Tiếng Việt**

## 10. Test cuối (không dùng để chọn model)
**Phụ trách: Member B.** Gọi: `evaluate(model, test_loader, device)` chỉ cho cấu hình đã chọn ở mục 7 và 9.

In [ ]:
# Owner: Member B
from labs.lab2.evaluate.evaluate import evaluate

if "model" in locals() and "test_loader" in locals():
    test_loss, test_acc = evaluate(model, test_loader, device)
    print(f"Final Test Loss: {test_loss:.4f}")
    print(f"Final Test Accuracy: {test_acc * 100:.2f}%")
else:
    print("[NOTE] model or test_loader not initialized. Final test will be evaluated after model selection.")


## 11. Summary
**Owner: team leader.** 3-5 modest sentences based only on the outputs above (to be written after the runs).

---

**Tiếng Việt**

## 11. Tổng kết
**Phụ trách: nhóm trưởng.** 3-5 câu khiêm tốn, chỉ dựa trên output phía trên (viết sau khi chạy).

## 12. Appendix
**Owner: team leader.** Extra work only if understood (per-class accuracy, confusion pairs). Out of scope: ensemble, TTA, Optuna, ViT, MoE.

---

**Tiếng Việt**

## 12. Phụ lục
**Phụ trách: nhóm trưởng.** Phần làm thêm chỉ khi hiểu (accuracy từng lớp, cặp nhầm lẫn). Ngoài phạm vi: ensemble, TTA, Optuna, ViT, MoE.